# Make Tables for GPS

Claire Punturieri  
September 5, 2026

In [ ]:
#| message: false
#| warning: false

library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

## Demographics (table 1)

Load in study_dates.

In [ ]:
study_dates <- read_csv(file.path(path_shared,"study_dates_gps.csv"),
                        show_col_types = FALSE)

Pull list of subject IDs with at least one month data, credible lapse reporting, and sufficient GPS data.

In [ ]:
subids_dates <- study_dates |>  
  pull(subid) |>  
  unique()

Clean up file and filter down to only subjects used for GPS.

In [ ]:
dem_data <- read_csv(file.path(path_shared, "screen.csv"),
                     show_col_types = FALSE) |> 
  select(subid,  
         age = dem_1, 
         sex = dem_2, 
         race = dem_3, 
         hispanic = dem_4,
         education = dem_5,
         work = dem_6,
         income = dem_7,
         contains("dsm")) |> 
         mutate(hispanic = if_else(str_detect(hispanic, "Yes"), "1", "0")) |>  
    mutate(american_native = ifelse(str_detect(race,"American Indian/Alaska Native"),1,0),
        asian = ifelse(str_detect(race,"Asian"),1,0),
        pacific = ifelse(str_detect(race,"Native Hawaiian or Other Pacific Islander"),1,0),
        black = ifelse(str_detect(race,"Black/African American"),1,0),
        white = ifelse(str_detect(race,"White/Caucasian"),1,0),
        other = ifelse(str_detect(race,"Other/Multiracial"),1,0),
        ) |> 
  filter(subid %in% subids_dates)

Create table.

In [ ]:
#| label: tbl-1
#| tbl-cap: "Demographic Characteristics"

dem_data |>
  mutate(
    age_group = cut(
      age,
      breaks = c(18, 25, 35, 45, 55, 65, Inf),
      labels = c(
        "18-24",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "65+"
      ),
      right = FALSE
    ),
    income_group = cut(
      income,
      breaks = c(0, 25000, 50000, 100000, 150000, Inf),
      labels = c(
        "<$25,000",
        "$25,000-$49,999",
        "$50,000-$99,999",
        "$100,000-$149,999",
        "≥$150,000"
      ),
    right = FALSE
  ),
  hispanic = if_else(hispanic == 1, "Yes", "No")) |>
  gtsummary::tbl_summary(include = c(age_group, sex,
                                     education, work,
                                     race, hispanic, income_group),
                         label = list(
                           age_group ~ "Age",
                           sex ~ "Sex At Birth",
                           race ~ "Race",
                           hispanic ~ "Ethnicity",
                           income_group ~ "Income"
                         )) |> 
  gtsummary::bold_labels()

Characteristic,N = 1461
Age,
18-24,10 (6.8%)
25-34,40 (27%)
35-44,41 (28%)
45-54,35 (24%)
55-64,17 (12%)
65+,3 (2.1%)
Sex At Birth,
Female,72 (49%)
Male,74 (51%)


## Fairness by auROC (table 2)

In [ ]:
contrast_dem_auroc <- read_csv((here::here(path_models, "pp_fairness_contrast_full.csv")))

Rows: 5 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): contrast
dbl (4): probability, median, lower, upper

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

In [ ]:
#| label: tbl-2
#| tbl-cap: "Bayesian comparisons of median auROCs by demographic subgroup"

contrast_dem_auroc |> 
  mutate(`Bayesian CI` = str_c("[", round(lower, 3), ", ", round(upper, 3), "]"),
         `Median auROC Difference` = as.character(round(median, 3)),
         Probability = sprintf("%.3f", probability)) |> 
  select(Contrast = contrast, `Median auROC Difference`, `Bayesian CI`, Probability) |> 
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::column_spec(1, width = "25em") |> 
  kableExtra::add_footnote(#label = footnote_table_model,
                           notation = "none",
                           escape = FALSE) 

Contrast,Median auROC Difference,Bayesian CI,Probability
Male vs Female,0.017,"[-0.013, 0.048]",0.169
White/Caucasian vs Non-White and/or Hispanic,0.039,"[-0.015, 0.094]",0.119
Above 2018 Poverty Line vs Below 2018 Poverty Line,-0.016,"[-0.047, 0.014]",0.810
Younger than 55 vs Older than 55,0.078,"[0.034, 0.125]",0.002
College or more vs Less than college,-0.053,"[-0.087, -0.019]",0.994


## Fairness by LL (table 3)

In [ ]:
contrast_dem_ll <- read_csv((here::here(path_models, "pp_fairness_contrast_logloss.csv")))

Rows: 5 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): contrast
dbl (4): probability, median, lower, upper

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

In [ ]:
#| label: tbl-3
#| tbl-cap: "Bayesian comparisons of median mean Log Loss by demographic subgroup"

contrast_dem_ll |> 
  mutate(`Bayesian CI` = str_c("[", round(lower, 3), ", ", round(upper, 3), "]"),
         `Median mean Log Loss Difference` = as.character(round(median, 3)),
         Probability = sprintf("%.3f", probability)) |> 
  select(Contrast = contrast, `Median mean Log Loss Difference`, `Bayesian CI`, Probability) |> 
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::column_spec(1, width = "25em") |> 
  kableExtra::add_footnote(#label = footnote_table_model,
                           notation = "none",
                           escape = FALSE) 

Contrast,Median mean Log Loss Difference,Bayesian CI,Probability
Male vs Female,-0.057,"[-0.085, -0.03]",1.000
White/Caucasian vs Non-White and/or Hispanic,0.052,"[0, 0.103]",0.051
Above 2018 Poverty Line vs Below 2018 Poverty Line,-0.05,"[-0.074, -0.027]",1.000
Younger than 55 vs Older than 55,-0.028,"[-0.079, 0.022]",0.819
College or more vs Less than college,0.019,"[-0.004, 0.043]",0.087


## Bin counts for calibration plots (tables 4 + 5)

In [ ]:
probs <- read_csv(here::here(path_models, "outer_preds_v25_nested_2_x_5_6_x_5_full.csv"))

Rows: 68616 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): label
dbl (4): id_obs, outer_split_num, prob_raw, prev_inner

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

Table 4 - collapsed across folds.

In [ ]:
#| label: tbl-4
#| tbl-cap: "Bayesian comparisons of median mean Log Loss by demographic subgroup"

cal_raw |>
  group_by(bins, probs) |>
  summarise(
    mean_pred = mean(prob),
    mean_lapse = mean(lapse),
    n = n(),
    .groups = "drop"
  ) |>
  mutate(mean_pred = (round(mean_pred, digits = 3) * 100),
         mean_lapse = (round(mean_lapse, digits = 3)) * 100) |> 
  select(`Bin width` = bins, `Mean Probability of Lapse` = mean_pred,
         `Mean Lapse Percentage` = mean_lapse, Count = n) |>
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::column_spec(1, width = "25em") |> 
  kableExtra::add_footnote(#label = footnote_table_model,
                           notation = "none",
                           escape = FALSE) 

Bin width,Mean Probability of Lapse,Mean Lapse Percentage,Count
"(0,0.1]",6.4,4.9,52192
"(0.1,0.2]",12.8,14.9,15771
"(0.2,0.3]",23.1,23.6,606
"(0.3,0.4]",32.3,21.7,46
"(0.4,0.5]",40.0,0.0,1


Table 5 - separated by fold.

In [ ]:
#| label: tbl-5
#| tbl-cap: "Bayesian comparisons of median mean Log Loss by demographic subgroup"

cal_raw |>
  group_by(bins, probs, outer_split_num) |>
  summarise(
    mean_pred = mean(prob),
    mean_lapse = mean(lapse),
    n = n(),
    .groups = "drop"
  ) |> 
  arrange(outer_split_num) |>
  mutate(mean_pred = (round(mean_pred, digits = 3) * 100),
         mean_lapse = (round(mean_lapse, digits = 3)) * 100) |> 
  select(`Bin width` = bins, `Fold` = outer_split_num,
         `Mean Probability of Lapse` = mean_pred,
         `Mean Lapse Percentage` = mean_lapse, Count = n) |>
  knitr::kable() |> 
  kable_classic() |> 
  kableExtra::column_spec(1, width = "25em") |> 
  kableExtra::add_footnote(#label = footnote_table_model,
                           notation = "none",
                           escape = FALSE) 

Bin width,Fold,Mean Probability of Lapse,Mean Lapse Percentage,Count
"(0,0.1]",1,6.5,5.1,1975
"(0.1,0.2]",1,12.2,13.8,356
"(0.2,0.3]",1,23.6,0.0,1
"(0,0.1]",2,7.6,4.2,1433
"(0.1,0.2]",2,13.7,9.3,841
"(0.2,0.3]",2,23.1,33.3,3
"(0,0.1]",3,7.7,4.5,1477
"(0.1,0.2]",3,12.6,17.1,692
"(0.2,0.3]",3,23.9,4.8,21
"(0,0.1]",4,5.6,6.2,2117
